# 2-2단계. 사람 검수표 품질점수 계산

> 2단계에서 만든 40개 검수표 중 `COMPLETED` 행만 사용해 전사·역할·자동 추출 품질을 계산합니다.

- 입력: `human_review_sample_40_final.xlsx` 또는 `human_review_sample_40.xlsx`
- 출력: 품질지표, 행별 채점 결과, 잘못 입력된 값, 역할 혼동행렬, 간단 보고서
- 주의: `IN_PROGRESS`와 `NOT_REVIEWED` 행은 점수에서 제외합니다.

In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas openpyxl jiwer scikit-learn seaborn matplotlib

In [ ]:
# 1. 라이브러리 불러오기 및 Google Drive 연결
from google.colab import drive
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from jiwer import cer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)

drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로 설정과 검수표 확인

In [ ]:
# 2. 경로 설정
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
REVIEW_ROOT = PROJECT_ROOT / '데이터셋 품질테스트_v3'
SCORE_ROOT = REVIEW_ROOT / 'quality_scoring'
SCORE_ROOT.mkdir(parents=True, exist_ok=True)

# final 파일이 있으면 우선 사용하고, 없으면 사용자가 이름을 바꾼 기본 파일을 사용합니다.
review_candidates = [
    REVIEW_ROOT / 'human_review_sample_40_final.xlsx',
    REVIEW_ROOT / 'human_review_sample_40.xlsx',
]
REVIEW_PATH = next((path for path in review_candidates if path.exists()), None)

assert REVIEW_PATH is not None, (
    '검수표를 찾지 못했습니다. 데이터셋 품질테스트_v3 폴더에 '
    'human_review_sample_40_final.xlsx 또는 human_review_sample_40.xlsx를 넣으세요.'
)
print('검수표:', REVIEW_PATH)
print('결과 저장:', SCORE_ROOT)

In [ ]:
# 3. 검수표 불러오기
excel_file = pd.ExcelFile(REVIEW_PATH)
sheet_name = 'review_sample' if 'review_sample' in excel_file.sheet_names else excel_file.sheet_names[-1]
review_df = pd.read_excel(REVIEW_PATH, sheet_name=sheet_name)
review_df.columns = review_df.columns.astype(str).str.strip()

required_columns = [
    'review_no', 'raw_text', 'auto_role', 'review_status', 'gold_text', 'gold_role',
    'impersonation_correct', 'action_correct', 'strategy_correct', 'amount_correct'
]
missing_columns = [column for column in required_columns if column not in review_df.columns]
assert not missing_columns, f'필수 컬럼이 없습니다: {missing_columns}'

review_df['review_status_normalized'] = (
    review_df['review_status'].fillna('').astype(str).str.strip().str.upper()
)
completed_df = review_df[review_df['review_status_normalized'] == 'COMPLETED'].copy()

print('전체 표본:', len(review_df), '개')
print('검수 완료:', len(completed_df), '개')
print('점수 제외:', len(review_df) - len(completed_df), '개')
assert len(completed_df) > 0, (
    'COMPLETED 행이 없습니다. 사람이 확인한 행의 review_status를 COMPLETED로 변경하세요.'
)
display(completed_df[['review_no', 'raw_text', 'auto_role', 'gold_text', 'gold_role']].head())

## 2. 입력값 검사

`TRUE`, `FALSE`만 정답률에 포함하며 `UNSURE`, `NOT_APPLICABLE`은 제외합니다. 그 밖의 값은 오류 목록에 저장합니다.

In [ ]:
# 4. 검수값 표준화와 잘못 입력된 값 확인
correctness_columns = [
    'impersonation_correct', 'action_correct', 'strategy_correct', 'amount_correct'
]
allowed_correctness = {'TRUE', 'FALSE', 'UNSURE', 'NOT_APPLICABLE'}
allowed_roles = {'OFFENDER', 'VICTIM', 'THIRD_PARTY', 'REVIEW', 'EXCLUDE'}
invalid_rows = []

completed_df['gold_role'] = completed_df['gold_role'].fillna('').astype(str).str.strip().str.upper()
completed_df['auto_role'] = completed_df['auto_role'].fillna('').astype(str).str.strip().str.upper()

for row_index, row in completed_df.iterrows():
    if row['gold_role'] not in allowed_roles:
        invalid_rows.append({
            'review_no': row['review_no'], 'column': 'gold_role',
            'value': row['gold_role'], 'reason': '허용되지 않은 역할값'
        })
    for column in correctness_columns:
        value = str(row[column]).strip().upper() if pd.notna(row[column]) else ''
        completed_df.at[row_index, column] = value
        if value not in allowed_correctness:
            invalid_rows.append({
                'review_no': row['review_no'], 'column': column,
                'value': value, 'reason': 'TRUE/FALSE/UNSURE/NOT_APPLICABLE 중 하나를 입력'
            })

invalid_df = pd.DataFrame(invalid_rows, columns=['review_no', 'column', 'value', 'reason'])
invalid_df.to_csv(SCORE_ROOT / 'invalid_review_entries.csv', index=False, encoding='utf-8-sig')
print('잘못 입력된 값:', len(invalid_df), '개')
if len(invalid_df):
    display(invalid_df)
else:
    print('입력값 검사 통과')

## 3. 전사 정확도 계산

띄어쓰기와 문장부호를 제거한 뒤 `raw_text`와 `gold_text`의 문자 오류율(CER)을 계산합니다. 점수가 높을수록 정확합니다.

In [ ]:
# 5. 전사 CER 계산
def normalize_for_cer(text):
    text = unicodedata.normalize('NFKC', str(text or '')).lower()
    return re.sub(r'[^0-9a-z가-힣]', '', text)

text_eval_df = completed_df[
    (completed_df['gold_role'] != 'EXCLUDE') &
    completed_df['gold_text'].fillna('').astype(str).str.strip().ne('')
].copy()
text_eval_df['normalized_raw_text'] = text_eval_df['raw_text'].map(normalize_for_cer)
text_eval_df['normalized_gold_text'] = text_eval_df['gold_text'].map(normalize_for_cer)
text_eval_df = text_eval_df[text_eval_df['normalized_gold_text'].str.len() > 0].copy()
assert len(text_eval_df) > 0, 'CER을 계산할 gold_text가 없습니다.'

text_eval_df['cer'] = text_eval_df.apply(
    lambda row: cer(row['normalized_gold_text'], row['normalized_raw_text']), axis=1
)
corpus_cer = cer(
    text_eval_df['normalized_gold_text'].tolist(),
    text_eval_df['normalized_raw_text'].tolist(),
)
transcription_accuracy = max(0.0, 1.0 - corpus_cer)
print(f'전사 CER: {corpus_cer:.4f}')
print(f'전사 정확도: {transcription_accuracy * 100:.2f}점')

## 4. 범인·피해자 역할 분류 평가

In [ ]:
# 6. 역할 분류 Accuracy·Precision·Recall·F1 계산
role_labels = ['OFFENDER', 'VICTIM', 'THIRD_PARTY']
role_eval_df = completed_df[completed_df['gold_role'].isin(role_labels)].copy()
assert len(role_eval_df) > 0, '역할을 평가할 GOLD 행이 없습니다.'

y_true = role_eval_df['gold_role']
y_pred = role_eval_df['auto_role']
role_accuracy = accuracy_score(y_true, y_pred)
role_precision, role_recall, role_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=role_labels, average='macro', zero_division=0
)
role_report_df = pd.DataFrame(classification_report(
    y_true, y_pred, labels=role_labels, output_dict=True, zero_division=0
)).transpose().reset_index().rename(columns={'index': '역할'})
display(role_report_df)
print(f'역할 정확도: {role_accuracy * 100:.2f}점')
print(f'역할 Macro F1: {role_f1:.4f}')

In [ ]:
# 7. 역할 혼동행렬 저장
matrix_labels = role_labels + ([
    'REVIEW'
] if 'REVIEW' in set(y_pred) else [])
role_cm = confusion_matrix(y_true, y_pred, labels=matrix_labels)
plt.figure(figsize=(7, 5))
sns.heatmap(role_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=matrix_labels, yticklabels=matrix_labels)
plt.title('Role confusion matrix')
plt.xlabel('Predicted role')
plt.ylabel('Gold role')
plt.tight_layout()
plt.savefig(SCORE_ROOT / 'role_confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. 사칭·행동·심리전략·금액 추출 정답률

현재 검수표에는 실제 정답 라벨이 아니라 자동 판정의 정오가 기록되므로, 이 네 항목은 Precision·Recall이 아닌 표본 정답률을 계산합니다.

In [ ]:
# 8. 항목별 정답률 계산
correctness_names = {
    'impersonation_correct': '사칭 대상 판정',
    'action_correct': '요구 행동 판정',
    'strategy_correct': '심리전략 판정',
    'amount_correct': '금액 추출 판정',
}
correctness_rows = []
for column, korean_name in correctness_names.items():
    valid = completed_df[column].isin(['TRUE', 'FALSE'])
    evaluated = completed_df.loc[valid, column]
    correct_count = int((evaluated == 'TRUE').sum())
    incorrect_count = int((evaluated == 'FALSE').sum())
    accuracy = correct_count / len(evaluated) if len(evaluated) else np.nan
    correctness_rows.append({
        '평가항목': korean_name, '평가건수': len(evaluated),
        '정답건수': correct_count, '오답건수': incorrect_count,
        '정답률': accuracy, '점수_100점': accuracy * 100 if pd.notna(accuracy) else np.nan,
    })
correctness_df = pd.DataFrame(correctness_rows)
display(correctness_df)

## 6. 결과 저장과 최종 확인

In [ ]:
# 9. 품질지표 요약
metrics_rows = [
    {'평가항목': '검수 완료 발화', '값': len(completed_df), '표시값': f'{len(completed_df)}건'},
    {'평가항목': '전사 CER', '값': corpus_cer, '표시값': f'{corpus_cer:.4f}'},
    {'평가항목': '전사 정확도', '값': transcription_accuracy, '표시값': f'{transcription_accuracy * 100:.2f}점'},
    {'평가항목': '역할 분류 정확도', '값': role_accuracy, '표시값': f'{role_accuracy * 100:.2f}점'},
    {'평가항목': '역할 Macro Precision', '값': role_precision, '표시값': f'{role_precision:.4f}'},
    {'평가항목': '역할 Macro Recall', '값': role_recall, '표시값': f'{role_recall:.4f}'},
    {'평가항목': '역할 Macro F1', '값': role_f1, '표시값': f'{role_f1:.4f}'},
]
for row in correctness_rows:
    metrics_rows.append({
        '평가항목': row['평가항목'] + ' 정답률',
        '값': row['정답률'],
        '표시값': f"{row['점수_100점']:.2f}점" if pd.notna(row['점수_100점']) else '평가 불가',
    })
metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df[['평가항목', '표시값']])

In [ ]:
# 10. 행별 결과와 보고서 저장
per_row_df = completed_df.copy()
per_row_df = per_row_df.merge(
    text_eval_df[['review_no', 'normalized_raw_text', 'normalized_gold_text', 'cer']],
    on='review_no', how='left'
)
per_row_df['role_correct'] = np.where(
    per_row_df['gold_role'].isin(role_labels),
    per_row_df['auto_role'] == per_row_df['gold_role'],
    np.nan,
)

metrics_df.to_csv(SCORE_ROOT / 'quality_metrics.csv', index=False, encoding='utf-8-sig')
correctness_df.to_csv(SCORE_ROOT / 'event_correctness_scores.csv', index=False, encoding='utf-8-sig')
role_report_df.to_csv(SCORE_ROOT / 'role_classification_report.csv', index=False, encoding='utf-8-sig')
per_row_df.to_csv(SCORE_ROOT / 'review_scored_rows.csv', index=False, encoding='utf-8-sig')

report_lines = [
    '# 보이스피싱 데이터셋 표본 품질평가 결과', '',
    f'- 평가 파일: `{REVIEW_PATH.name}`',
    f'- 전체 표본: {len(review_df)}건',
    f'- 검수 완료 및 채점 대상: {len(completed_df)}건',
    f'- 전사 CER: {corpus_cer:.4f}',
    f'- 전사 정확도: {transcription_accuracy * 100:.2f}점',
    f'- 역할 분류 정확도: {role_accuracy * 100:.2f}점',
    f'- 역할 Macro F1: {role_f1:.4f}', '',
    '## 자동 추출 항목별 정답률', '',
]
for row in correctness_rows:
    score_text = f"{row['점수_100점']:.2f}점" if pd.notna(row['점수_100점']) else '평가 불가'
    report_lines.append(f"- {row['평가항목']}: {score_text} ({row['평가건수']}건)")
report_lines += [
    '', '## 해석 시 주의사항', '',
    '- 이 결과는 40개 이하의 표본 품질평가이며 전체 데이터의 정확도를 확정하지 않습니다.',
    '- COMPLETED로 표시된 행만 계산에 포함했습니다.',
    '- 사칭·행동·전략·금액은 현재 검수표 구조상 Precision·Recall이 아닌 정답률입니다.',
]
(SCORE_ROOT / 'quality_scoring_report.md').write_text('\n'.join(report_lines), encoding='utf-8')

summary = {
    'review_file': REVIEW_PATH.name,
    'total_rows': int(len(review_df)),
    'completed_rows': int(len(completed_df)),
    'invalid_entries': int(len(invalid_df)),
    'transcription_cer': float(corpus_cer),
    'transcription_accuracy': float(transcription_accuracy),
    'role_accuracy': float(role_accuracy),
    'role_macro_f1': float(role_f1),
}
(SCORE_ROOT / 'quality_scoring_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('채점 결과 저장 완료:', SCORE_ROOT)

In [ ]:
# 11. 최종 산출물 확인
expected_files = [
    'quality_metrics.csv',
    'event_correctness_scores.csv',
    'role_classification_report.csv',
    'review_scored_rows.csv',
    'invalid_review_entries.csv',
    'role_confusion_matrix.png',
    'quality_scoring_report.md',
    'quality_scoring_summary.json',
]
missing_outputs = [name for name in expected_files if not (SCORE_ROOT / name).exists()]
assert not missing_outputs, f'생성되지 않은 결과가 있습니다: {missing_outputs}'
print('정상 완료: 산출물', len(expected_files), '개')
for name in expected_files:
    print('-', name)

## 실행 결과

결과는 다음 폴더에 저장됩니다.

```text
내 드라이브/보이스피싱_분석/데이터셋 품질테스트_v2/quality_scoring/
```

점수는 표본에 대한 품질 확인 결과이며 전체 데이터의 확정 정확도로 표현하지 않습니다.